# FrogPath – Rate My Professor Data Exploration

**Course:** Digital Culture and Data Analytics Capstone  
**Goal:** Load, clean, and explore RMP data for TCU instructors; generate
summary statistics and visualisations that feed the FrogPath web app.

### Contents
1. Setup & imports  
2. Load raw data  
3. Clean & validate  
4. Summary statistics  
5. Visualisations  
   - Rating distribution  
   - Average rating by department  
   - Average difficulty by department  
   - Rated vs. unrated coverage  
6. Export JSON outputs  

## 1 · Setup & imports

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# Paths (relative to this notebook in analysis/)
NOTEBOOK_DIR = os.path.abspath(".")          # …/analysis
BASE_DIR     = os.path.dirname(NOTEBOOK_DIR) # …/Finalproject
DATA_DIR     = os.path.join(BASE_DIR, "data", "raw")
OUT_DIR      = os.path.join(BASE_DIR, "outputs")
FIG_DIR      = os.path.join(OUT_DIR, "figures")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RMP_CSV  = os.path.join(DATA_DIR, "rmp_tcu_sample.csv")
DEPT_CSV = os.path.join(DATA_DIR, "tcu_course_catalog_sample.csv")

print(f"Data directory : {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")

## 2 · Load raw data

In [ ]:
rmp_raw = pd.read_csv(RMP_CSV)
catalog = pd.read_csv(DEPT_CSV)

print(f"RMP rows: {len(rmp_raw)} | columns: {list(rmp_raw.columns)}")
rmp_raw.head()

In [ ]:
print(f"Catalog rows: {len(catalog)} | columns: {list(catalog.columns)}")
catalog.head()

## 3 · Clean & validate

In [ ]:
rmp = rmp_raw.copy()

# Normalise column names
rmp.columns = [c.strip().lower().replace(" ", "_") for c in rmp.columns]

# Cast numeric columns
numeric_cols = ["overall_rating", "difficulty", "num_ratings", "would_take_again"]
for col in numeric_cols:
    if col in rmp.columns:
        rmp[col] = pd.to_numeric(rmp[col], errors="coerce")

# Drop rows missing key fields
rmp.dropna(subset=["professor_name", "department", "overall_rating"], inplace=True)

# Strip whitespace from strings
rmp["professor_name"] = rmp["professor_name"].str.strip()
rmp["department"]     = rmp["department"].str.strip()

# Keep valid rating ranges
rmp = rmp[(rmp["overall_rating"].between(0, 5)) &
          (rmp["difficulty"].between(0, 5))]

rmp.reset_index(drop=True, inplace=True)
print(f"Rows after cleaning: {len(rmp)}")
rmp.info()

In [ ]:
rmp.describe()

## 4 · Summary statistics

In [ ]:
# Average rating per department
ratings_by_dept = (
    rmp.groupby("department")["overall_rating"]
       .agg(avg_rating="mean", count="count")
       .round(2)
       .reset_index()
       .sort_values("avg_rating", ascending=False)
)
print("=== Average Overall Rating by Department ===")
ratings_by_dept

In [ ]:
# Average difficulty per department
difficulty_by_dept = (
    rmp.groupby("department")["difficulty"]
       .agg(avg_difficulty="mean", count="count")
       .round(2)
       .reset_index()
       .sort_values("avg_difficulty", ascending=False)
)
print("=== Average Difficulty Score by Department ===")
difficulty_by_dept

In [ ]:
# RMP coverage vs. catalog departments
catalog_depts = set(catalog["department"].str.strip().unique())
rmp_depts     = set(rmp["department"].str.strip().unique())
covered       = catalog_depts & rmp_depts
uncovered     = catalog_depts - rmp_depts

print(f"Total departments in catalog : {len(catalog_depts)}")
print(f"Departments with RMP data    : {len(covered)}")
print(f"Departments WITHOUT RMP data : {len(uncovered)}")
if uncovered:
    print(f"  Missing: {sorted(uncovered)}")
print(f"\nTotal rated professors       : {rmp['professor_name'].nunique()}")

## 5 · Visualisations

In [ ]:
# 5a – Rating distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(rmp["overall_rating"], bins=20, color="#4e2a84", edgecolor="white")
ax.axvline(rmp["overall_rating"].mean(), color="#a3c4f3", linestyle="--",
           label=f"Mean: {rmp['overall_rating'].mean():.2f}")
ax.set_title("Distribution of Overall Ratings – TCU Instructors (RMP)", fontsize=14)
ax.set_xlabel("Overall Rating (1–5)", fontsize=12)
ax.set_ylabel("Number of Professors", fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "rating_dist.png"), dpi=150)
plt.show()

In [ ]:
# 5b – Average rating by department (horizontal bar)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(ratings_by_dept["department"], ratings_by_dept["avg_rating"],
               color="#4e2a84", edgecolor="white")
ax.set_title("Average Overall Rating by Department", fontsize=14)
ax.set_xlabel("Average Rating (1–5)", fontsize=12)
ax.set_xlim(0, 5)
for bar, val in zip(bars, ratings_by_dept["avg_rating"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "dept_ratings.png"), dpi=150)
plt.show()

In [ ]:
# 5c – Average difficulty by department
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(difficulty_by_dept["department"], difficulty_by_dept["avg_difficulty"],
               color="#a3c4f3", edgecolor="white")
ax.set_title("Average Difficulty Score by Department", fontsize=14)
ax.set_xlabel("Average Difficulty (1–5)", fontsize=12)
ax.set_xlim(0, 5)
for bar, val in zip(bars, difficulty_by_dept["avg_difficulty"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "dept_difficulty.png"), dpi=150)
plt.show()

In [ ]:
# 5d – Rated vs. unrated departments (pie chart)
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [len(covered), len(uncovered)],
    labels=["Covered by RMP", "Not on RMP"],
    colors=["#4e2a84", "#d3d3d3"],
    autopct="%1.0f%%",
    startangle=140
)
ax.set_title("Department RMP Coverage", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "coverage_pie.png"), dpi=150)
plt.show()

## 6 · Export JSON outputs

In [ ]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2)
    print(f"Saved → {path}")

save_json(rmp.to_dict(orient="records"),
          os.path.join(OUT_DIR, "rmp_clean.json"))

save_json(ratings_by_dept.to_dict(orient="records"),
          os.path.join(OUT_DIR, "dept_avg_ratings.json"))

save_json(difficulty_by_dept.to_dict(orient="records"),
          os.path.join(OUT_DIR, "dept_avg_difficulty.json"))

coverage_data = {
    "rated_professors":       int(rmp["professor_name"].nunique()),
    "departments_covered":    len(covered),
    "departments_uncovered":  len(uncovered),
    "covered_list":           sorted(covered),
    "uncovered_list":         sorted(uncovered),
}
save_json(coverage_data,
          os.path.join(OUT_DIR, "rated_vs_unrated.json"))

print("\nAll outputs written successfully.")